# Embedding demo

(Replaces the embedding half of `scripts/embedding_norm_demo.py`. Self-contained: only external libraries.)

The first step of the transformer. Each token id is replaced by a learned vector of `hidden_size` numbers.
It is a plain table lookup, with no arithmetic: the table has one row per vocabulary entry, and token `i`
gets row `i`. The rows start random and are adjusted during training so that similar tokens end up with
similar vectors.

We use random weights (std 0.02, like the real model's init), so the numbers mean nothing yet. The point is
the shapes and the behaviour.

Select the kernel **smalllm (.venv)** before running.

In [1]:
import torch
from torch import nn

torch.manual_seed(0)

VOCAB_SIZE = 49152  # SmolLM2 tokenizer
HIDDEN_SIZE = 128   # the "nano" size
IDS = [6403, 1980, 253, 655]  # "Once upon a time" (see tokenize_demo.ipynb)

## 1. The table
A matrix of shape `(vocab, hidden)`, initialised with small random numbers.

In [2]:
embed = nn.Embedding(VOCAB_SIZE, HIDDEN_SIZE)
nn.init.normal_(embed.weight, mean=0.0, std=0.02)
print(f"table shape: {tuple(embed.weight.shape)}  (vocab, hidden)")
print(f"parameters:  {embed.weight.numel():,}")

table shape: (49152, 128)  (vocab, hidden)
parameters:  6,291,456


## 2. Look up four tokens
The input is a batch of one sentence of four ids, shape `(1, 4)`. The output adds a `hidden` dimension.

In [3]:
ids = torch.tensor([IDS])
vectors = embed(ids)
print(f"ids shape: {tuple(ids.shape)}  ->  vectors shape: {tuple(vectors.shape)}")

ids shape: (1, 4)  ->  vectors shape: (1, 4, 128)


## 3. It really is just a row lookup
Token 6403 gets row 6403 of the table, exactly.

In [4]:
print(f"is row {IDS[0]} of the table: {torch.equal(vectors[0, 0], embed.weight[IDS[0]])}")
print(f"first 6 numbers of token {IDS[0]}: {[round(v, 4) for v in vectors[0, 0, :6].tolist()]}")

is row 6403 of the table: True
first 6 numbers of token 6403: [-0.0016, -0.019, -0.0036, 0.001, -0.0276, 0.0267]


Equivalent view: a one-hot vector times the table (a matrix multiply that just picks one row). The lookup is the fast way to do the same thing.

In [5]:
one_hot = torch.zeros(VOCAB_SIZE)
one_hot[IDS[0]] = 1.0
print("one-hot @ table equals the lookup:", torch.allclose(one_hot @ embed.weight, vectors[0, 0]))

one-hot @ table equals the lookup: True


## 4. Typical size of each vector
Root-mean-square of each vector. With std-0.02 init it is about 0.02. This is the scale RMSNorm will later fix (see `rmsnorm_demo.ipynb`).

In [6]:
rms = vectors.pow(2).mean(-1).sqrt()[0].tolist()
print("rms of each token vector:", [round(v, 4) for v in rms])

rms of each token vector: [0.0182, 0.0183, 0.0194, 0.0201]


## 5. Same id, same vector
The embedding knows nothing about position or neighbours: the same token always gets the same vector.
Position is added later by RoPE inside attention.

In [7]:
twice = embed(torch.tensor([[253, 655, 253]]))
print("token 253 at position 0 equals token 253 at position 2:", torch.equal(twice[0, 0], twice[0, 2]))

token 253 at position 0 equals token 253 at position 2: True


## 6. Weight tying
In the full model the output layer reuses this same table (transposed) to turn the final vectors back into
a score per vocabulary entry. That halves the parameters spent on the vocabulary. Here is the idea:

In [8]:
lm_head = nn.Linear(HIDDEN_SIZE, VOCAB_SIZE, bias=False)
lm_head.weight = embed.weight  # one shared Parameter

hidden_state = torch.randn(1, 1, HIDDEN_SIZE)
print("logits shape:", tuple(lm_head(hidden_state).shape), "(one score per vocabulary entry)")
print("shared with the embedding:", lm_head.weight is embed.weight)
print("parameters counted once:", sum(p.numel() for p in nn.ModuleList([embed, lm_head]).parameters()))

logits shape: (1, 1, 49152) (one score per vocabulary entry)
shared with the embedding: True
parameters counted once: 6291456
